# Dora Maar — Artnet Image Scraper

Scrapes all Dora Maar artworks from Artnet using a visible Firefox browser, downloads every thumbnail image, and saves a metadata CSV with file paths.

In [ ]:
import os, ssl, certifi
os.environ["SSL_CERT_FILE"]      = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()
ssl._create_default_https_context = ssl.create_default_context

import re
import time
import pandas as pd
import browser_cookie3
from pathlib import Path
from bs4 import BeautifulSoup
from curl_cffi import requests as curl_requests

IMAGES_DIR = Path("images")
IMAGES_DIR.mkdir(exist_ok=True)

print("Setup complete. Images will be saved to:", IMAGES_DIR.resolve())

---
## Step 1 — Fetch all 64 Artnet pages

**Before running this cell:**  
1. Open Firefox manually and go to `https://www.artnet.com/artists/dora-maar/`  
2. Solve any CAPTCHA until artworks are visible  
3. Come back here and run the cell — cookies are read directly from your Firefox profile

In [ ]:
TOTAL_PAGES = 64
BASE_URL    = "https://www.artnet.com/artists/dora-maar/{}"

# --- Read cookies straight from your Firefox profile (no Selenium needed) ---
ff_cookies = browser_cookie3.firefox(domain_name=".artnet.com")
cookie_list = list(ff_cookies)
print(f"Read {len(cookie_list)} artnet.com cookies from Firefox profile:\n")
for c in cookie_list:
    print(f"  {c.name:40s}  {c.value[:60]}")

incapsula = [c for c in cookie_list if "incap" in c.name.lower() or "visid" in c.name.lower()]
print(f"\nIncapsula cookies found: {len(incapsula)}")
for c in incapsula:
    print(f"  {c.name} = {c.value[:80]}")

if not incapsula:
    print("\n⚠  No Incapsula cookies found.")
    print("   Make sure you opened artnet.com in Firefox and solved any CAPTCHA before running this cell.")

# --- curl_cffi session: Firefox TLS fingerprint + your real browser cookies ---
session = curl_requests.Session(impersonate="firefox135")
for c in cookie_list:
    session.cookies.set(c.name, c.value, domain=c.domain)

print(f"\nSession ready. Fetching {TOTAL_PAGES} pages...\n")

# --- Fetch all 64 pages ---
html_pages = {}

for page in range(1, TOTAL_PAGES + 1):
    url = BASE_URL.format(page)
    try:
        resp = session.get(url, timeout=15)
        size = len(resp.text)
        blocked = "Incapsula" in resp.text or size < 5_000
        status = "BLOCKED ⚠" if blocked else "ok"
        print(f"  Page {page:2d}/{TOTAL_PAGES}  {resp.status_code}  {size:>8,} chars  [{status}]")
        html_pages[page] = "" if blocked else resp.text
    except Exception as e:
        print(f"  Page {page:2d}/{TOTAL_PAGES}  ERROR: {e}")
        html_pages[page] = ""
    if page < TOTAL_PAGES:
        time.sleep(5)

ok = sum(1 for h in html_pages.values() if h)
print(f"\nDone. {ok}/{TOTAL_PAGES} pages fetched successfully.")


In [ ]:
from artnet_html_parser import save_pages
# --- Save HTML to disk so Step 2 can be re-run without re-fetching ---
save_pages(html_pages, "artnet_html")

---
## Step 2 — Parse artworks & save catalog CSV

Loads HTML from `artnet_html/`, parses every artwork block, and saves `artnet_catalog_raw.csv`.  
Columns: `page, title, year, gallery_name, gallery_url, price, img_url, detail_url`

In [ ]:
html_pages[1]

In [ ]:
from artnet_html_parser import parse_pages, load_pages

CATALOG_RAW = Path("artnet_catalog_raw.csv")

html_pages = load_pages("artnet_html")
print(f"Loaded {len(html_pages)} HTML files from artnet_html/")

df = parse_pages(html_pages)
print(f"Parsed {len(df)} unique artworks")

df.to_csv(CATALOG_RAW, index=False)
print(f"Saved to {CATALOG_RAW.resolve()}")
df.head(10)

---
## Step 3 — Download images & save metadata CSV

Images saved to `images/` as `{index:04d}_{slug}.jpg`.  
Progress is written to `artnet_catalog.csv` every 50 downloads so the run can be resumed.

In [ ]:
CSV_PATH = Path("artnet_catalog.csv")

# Resume from existing CSV if present
if CSV_PATH.exists():
    df_done = pd.read_csv(CSV_PATH)
    done_urls = set(df_done["img_url"])
    print(f"Resuming — {len(df_done)} already downloaded.")
else:
    df_done = pd.DataFrame()
    done_urls = set()

records = []
img_session = curl_requests.Session(impersonate="firefox135")

for idx, row in df.iterrows():
    if row["img_url"] in done_urls:
        continue

    slug = re.sub(r"[^a-z0-9]+", "-", row["img_url"].split("/")[-1].lower())[:80]
    filename = f"{idx:04d}_{slug}"
    if not filename.endswith(".jpg"):
        filename += ".jpg"
    local_path = IMAGES_DIR / filename

    try:
        resp = img_session.get(row["img_url"], timeout=15)
        resp.raise_for_status()
        local_path.write_bytes(resp.content)
        status = "ok"
    except Exception as e:
        local_path = Path("")
        status = f"error: {e}"

    records.append({
        **row.to_dict(),
        "local_path": str(local_path),
        "status":     status,
    })

    if len(records) % 50 == 0:
        batch = pd.concat([df_done, pd.DataFrame(records)], ignore_index=True)
        batch.to_csv(CSV_PATH, index=False)
        print(f"  Checkpoint: {len(batch)} rows saved.")

    time.sleep(0.15)

df_catalog = pd.concat([df_done, pd.DataFrame(records)], ignore_index=True)
df_catalog.to_csv(CSV_PATH, index=False)

ok = (df_catalog["status"] == "ok").sum()
print(f"\nDone. {ok} / {len(df_catalog)} images downloaded.")
print(f"Catalog saved to: {CSV_PATH.resolve()}")
df_catalog.head(10)

---
## Step 4 — Scrape Picasso's Dora Maar artworks

Fetches the 62 artworks from `pablo-ruiz-picasso.net/theme-doramaar.php`, visits each detail page for clean title + year, downloads images to `images/picasso_dora/`, and saves `picasso_dora_catalog.csv`.

In [ ]:
import importlib, parse_html
importlib.reload(parse_html)
from parse_html import scrape_picasso_dora_maar

PICASSO_DIR = Path("images/picasso_dora")
PICASSO_DIR.mkdir(parents=True, exist_ok=True)
PICASSO_CSV = Path("picasso_dora_catalog.csv")

# --- Scrape listing + visit each detail page ---
pic_session = curl_requests.Session(impersonate="firefox135")
records = scrape_picasso_dora_maar(session=pic_session)
print(f"Scraped {len(records)} artworks from pablo-ruiz-picasso.net")

# --- Download full-size images ---
for i, rec in enumerate(records):
    slug       = re.sub(r"[^a-z0-9]+", "-", rec["img_url"].split("/")[-1].lower())[:60]
    local_path = PICASSO_DIR / f"{i:03d}_{slug}"
    if not local_path.suffix:
        local_path = local_path.with_suffix(".jpg")

    try:
        r = pic_session.get(rec["img_url"], timeout=15)
        r.raise_for_status()
        local_path.write_bytes(r.content)
        rec["local_path"] = str(local_path)
        rec["dl_status"]  = "ok"
    except Exception as e:
        rec["local_path"] = ""
        rec["dl_status"]  = f"error: {e}"

    if (i + 1) % 10 == 0:
        print(f"  downloaded {i+1}/{len(records)}…")

df_picasso = pd.DataFrame(records)
df_picasso.to_csv(PICASSO_CSV, index=False)

ok = (df_picasso["dl_status"] == "ok").sum()
print(f"\nDone. {ok}/{len(df_picasso)} images saved to {PICASSO_DIR}")
print(f"Catalog saved to: {PICASSO_CSV.resolve()}")
df_picasso.head(10)

---
## Step 5 — Scrape & download top Google Images results for "dora maar"

Fetches the first results page from `google.com/search?tbm=isch`, extracts ~90–100 image URLs using `parse_html.parse_google_images_html`, downloads each to `images/google_search/`, and saves `google_search_results.csv`.

**Run with Firefox open** (uses your Google cookies to avoid CAPTCHAs).

In [ ]:
import json, random
from urllib.parse import quote_plus
from parse_html import parse_google_images_html

GOOG_DIR = Path("images/google_search")
GOOG_DIR.mkdir(parents=True, exist_ok=True)
GOOG_CSV = Path("google_search_results.csv")

QUERY = "dora maar"

# --- Firefox cookies for Google session ---
goog_cookies = browser_cookie3.firefox(domain_name=".google.com")
goog_session = curl_requests.Session(impersonate="firefox135")
for c in goog_cookies:
    goog_session.cookies.set(c.name, c.value, domain=c.domain)

# --- Fetch the full Google Images results page ---
url  = f"https://www.google.com/search?q={quote_plus(QUERY)}&tbm=isch&hl=en"
resp = goog_session.get(url, timeout=20)
ranked = parse_google_images_html(resp.text)
print(f"status={resp.status_code}  html={len(resp.text):,} chars  images={len(ranked)}")

if not ranked:
    raise SystemExit("Step 4 aborted — no images extracted.")

# --- Resume from existing CSV (guard against empty file) ---
if GOOG_CSV.exists() and GOOG_CSV.stat().st_size > 0:
    df_goog_done = pd.read_csv(GOOG_CSV)
    done_urls    = set(df_goog_done["img_url"])
    print(f"Resuming — {len(df_goog_done)} already in CSV.")
else:
    df_goog_done = pd.DataFrame(columns=[
        "rank", "title", "source_domain", "source_url",
        "img_url", "thumbnail_url", "local_path", "dl_status", "lens_count",
    ])
    done_urls = set()

dl_session  = curl_requests.Session(impersonate="firefox135")
new_records = []

for rank, item in enumerate(ranked):
    if item["img_url"] in done_urls:
        continue

    ext        = re.search(r'\.(jpe?g|png|gif|webp)', item["img_url"], re.IGNORECASE)
    ext        = ("." + ext.group(1).lower()) if ext else ".jpg"
    slug       = re.sub(r"[^a-z0-9]+", "-", item["img_url"].split("/")[-1].lower())[:60]
    local_path = GOOG_DIR / f"{rank:04d}_{slug}{ext}"

    try:
        r = dl_session.get(item["img_url"], timeout=15)
        r.raise_for_status()
        local_path.write_bytes(r.content)
        dl_status = "ok"
    except Exception as e:
        local_path = Path("")
        dl_status  = f"error: {e}"

    new_records.append({
        "rank":          rank,
        "title":         item["title"],
        "source_domain": item["source_domain"],
        "source_url":    item["source_url"],
        "img_url":       item["img_url"],
        "thumbnail_url": item["thumbnail_url"],
        "local_path":    str(local_path),
        "dl_status":     dl_status,
        "lens_count":    "",
    })

    if len(new_records) % 50 == 0:
        batch = pd.concat([df_goog_done, pd.DataFrame(new_records)], ignore_index=True)
        batch.to_csv(GOOG_CSV, index=False)
        print(f"  Checkpoint: {len(batch)} rows saved.")

    time.sleep(random.uniform(0.3, 0.8))

df_goog = pd.concat([df_goog_done, pd.DataFrame(new_records)], ignore_index=True)
df_goog.to_csv(GOOG_CSV, index=False)

ok = (df_goog["dl_status"] == "ok").sum()
print(f"\nDone. {ok}/{len(df_goog)} images downloaded to {GOOG_DIR}")
print(f"Catalog saved to: {GOOG_CSV.resolve()}")
df_goog.head(10)

---
## Step 5b — Deduplicate Google Images results

Manual Dedup

In [ ]:
import imagehash
from PIL import Image

HASH_THRESHOLD = 8   # Hamming bits — 0=identical, 64=completely different

df_goog = pd.read_csv(GOOG_CSV)

# Drop rows whose image file was manually removed (user deleted Picasso/DM artworks)
alive = df_goog["local_path"].apply(lambda p: bool(p) and Path(p).exists())
n_removed = (~alive & df_goog["dl_status"].eq("ok")).sum()
df_goog = df_goog[alive | ~df_goog["dl_status"].eq("ok")].reset_index(drop=True)
print(f"Dropped {n_removed} rows with missing files → {len(df_goog)} rows remain")
df_goog.to_csv(GOOG_CSV, index=False)

ok_mask = df_goog["dl_status"] == "ok"
ok_rows = df_goog[ok_mask]

def phash_file(path):
    try:
        return imagehash.phash(Image.open(path))
    except Exception:
        return None

print(f"Hashing {len(ok_rows)} Google Images downloads…")
goog_hashes: dict[int, imagehash.ImageHash] = {}
for idx, row in ok_rows.iterrows():
    h = phash_file(row["local_path"])
    if h is not None:
        goog_hashes[idx] = h
print(f"  {len(goog_hashes)}/{len(ok_rows)} hashed")

# Within-Google dedup: mark later copy of any near-identical pair
is_dup: dict[int, bool] = {}
indices = list(goog_hashes.keys())

for i, idx_i in enumerate(indices):
    if is_dup.get(idx_i):
        continue
    h_i = goog_hashes[idx_i]
    for idx_j in indices[i + 1:]:
        if is_dup.get(idx_j):
            continue
        if h_i - goog_hashes[idx_j] <= HASH_THRESHOLD:
            is_dup[idx_j] = True

df_goog["is_dup"] = df_goog.index.map(lambda i: is_dup.get(i, False))
df_goog.to_csv(GOOG_CSV, index=False)

n_dup  = sum(is_dup.values())
n_keep = ok_mask.sum() - n_dup
print(f"Duplicates flagged: {n_dup}")
print(f"Unique kept:        {n_keep}")

---
## Step 6 — Google Lens exact-match count per image

For each downloaded image across all catalogs, uploads it to Google Lens and records the **exact matches** count (`udm=48`). Saves results back into each catalog's CSV. Rate-limited to ~10 s between requests; checkpoints every 10 images so the run is resumable.

In [3]:
import os, ssl, certifi
os.environ["SSL_CERT_FILE"]      = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()
ssl._create_default_https_context = ssl.create_default_context

import importlib, time, random, json
import pandas as pd
import parse_html
from pathlib import Path

importlib.reload(parse_html)
from parse_html import lens_exact_match_count, vision_web_count

# ── Choose scoring method ─────────────────────────────────────────────────────
METHOD = "vision"   # "lens"   → Google Lens scraping  → stores in lens_count
                  # "vision" → Cloud Vision API       → stores in vision_data + vision_score
# ─────────────────────────────────────────────────────────────────────────────

if METHOD == "lens":
    import browser_cookie3
    from curl_cffi import requests as curl_requests
    _session = curl_requests.Session(impersonate="firefox135")
    for c in browser_cookie3.firefox(domain_name=".google.com"):
        _session.cookies.set(c.name, c.value, domain=c.domain)
    col = "lens_count"
    def _score(path): return lens_exact_match_count(path, _session)
    def _store(df, idx, result):
        df.at[idx, "lens_count"] = result
    def _already_done(df):
        return pd.to_numeric(df["lens_count"], errors="coerce").ge(0).sum()
    def _not_scored(df):
        return df["lens_count"].isna() | df["lens_count"].eq("")
    def _sleep(): time.sleep(random.uniform(3, 8))
else:
    from google.cloud import vision as gv
    CREDENTIALS = Path("dora-494219-4123fdf73562.json")
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str(CREDENTIALS)
    _vision_client = gv.ImageAnnotatorClient()
    col = "vision_data"
    def _score(path): return vision_web_count(path, CREDENTIALS, client=_vision_client)
    def _store(df, idx, result):
        if isinstance(result, dict):
            df.at[idx, "vision_data"]  = json.dumps(result)
            df.at[idx, "vision_score"] = result["score"]
        else:
            df.at[idx, "vision_data"]  = result   # -2 error
            df.at[idx, "vision_score"] = result
    def _already_done(df):
        return df["vision_data"].notna().sum() - df["vision_data"].eq("").sum()
    def _not_scored(df):
        return df["vision_data"].isna() | df["vision_data"].eq("")
    def _sleep(): pass

CATALOGS = [
    {"csv": Path("google_search_results.csv"), "status_col": "dl_status", "skip_dup": True},
    {"csv": Path("picasso_dora_catalog.csv"),   "status_col": "dl_status", "skip_dup": False},
    {"csv": Path("artnet_catalog.csv"),         "status_col": "status",    "skip_dup": False},
]

total_scored = 0

for cat in CATALOGS:
    csv_path = cat["csv"]
    if not csv_path.exists():
        print(f"Skipping {csv_path.name} (not found)")
        continue

    df = pd.read_csv(csv_path)
    for c in ([col] if METHOD == "lens" else ["vision_data", "vision_score"]):
        if c not in df.columns:
            df[c] = ""
        df[c] = df[c].astype(object)

    mask = df[cat["status_col"]].eq("ok")
    if cat["skip_dup"] and "is_dup" in df.columns:
        mask &= ~df["is_dup"].eq(True)
    mask &= _not_scored(df)

    todo = df[mask]
    already = _already_done(df)
    print(f"\n{csv_path.name}: {len(todo)} to score  ({already} already done, method={METHOD})")

    for i, (idx, row) in enumerate(todo.iterrows()):
        result = _score(row["local_path"])
        _store(df, idx, result)
        total_scored += 1

        if METHOD == "lens":
            flag = "BLOCKED" if result == -1 else ("ERROR" if result == -2 else f"{result:,}")
        else:
            flag = "ERROR" if result == -2 else f"score={result['score']} (f={result['full_matching_images']} p={result['partial_matching_images']} pg={result['pages_with_matching_images']})"
        print(f"  [{i+1}/{len(todo)}] {Path(row['local_path']).name[:50]}  →  {flag}")

        if (i + 1) % 10 == 0:
            df.to_csv(csv_path, index=False)
            print(f"    ↳ checkpoint saved ({i+1} done in {csv_path.name})")

        _sleep()

    df.to_csv(csv_path, index=False)
    print(f"  → {_already_done(df)}/{len(df)} scored in {csv_path.name}")

print(f"\nTotal scored this run: {total_scored}")



google_search_results.csv: 19 to score  (0 already done, method=vision)
  [1/19] 0001_18-518309-web-crop-0-width-420-reqtjyl-jpg.jp  →  score=111 (f=32 p=7 pg=72)
  [2/19] 0004_aff5959af1adbbd7e6e2cfcb643b8ae16ad20742-1191  →  score=147 (f=34 p=44 pg=69)
  [3/19] 0005_-url-https-3a-2f-2fstudio-brooklynrail-org-2f  →  score=163 (f=54 p=24 pg=85)
  [4/19] 0009_doramaar5-jpg.jpg  →  score=81 (f=3 p=28 pg=50)
  [5/19] 0024_picasso-dora-maar-man-ray-jpg.jpg  →  score=89 (f=10 p=35 pg=44)
  [6/19] 0039_uh-bc-3180-1988-hidden-portraits-roe-mar-2025  →  score=136 (f=3 p=23 pg=110)
  [7/19] 0041_p07chpfy-jpg.jpg  →  score=98 (f=8 p=42 pg=48)
  [8/19] 0042_dora-maar-exhibition-catalog-jpg.jpg  →  score=178 (f=8 p=87 pg=83)
  [9/19] 0043_dm-in-door-jpg.jpg  →  score=0 (f=0 p=0 pg=0)
  [10/19] 0049_maison-dora-maar-jpg.jpg  →  score=15 (f=1 p=2 pg=12)
    ↳ checkpoint saved (10 done in google_search_results.csv)
  [11/19] 0050_dora-maar-0-jpg-itok-do63p2-i.jpg  →  score=40 (f=1 p=9 pg=30)
  [12/1

---
## Step 7 — Score distribution, top-40 histogram, and image collage

Three visualisations built from `vision_score` across all three catalogs:
1. **Distribution** — histogram + CDF of all scores
2. **Top-40 bar chart** — highest-scoring images coloured by catalog
3. **Treemap collage** — all 1 589 images; tile area ∝ score (zeros → white, area = 1)

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

# ── Load all catalogs ────────────────────────────────────────────────────────
def load_catalog(csv, status_col, skip_dup=False):
    df = pd.read_csv(csv)
    df = df[df[status_col].eq("ok")].copy()
    if skip_dup and "is_dup" in df.columns:
        df = df[~df["is_dup"].eq(True)]
    df["vision_score_num"] = pd.to_numeric(df.get("vision_score"), errors="coerce").fillna(0)
    return df

df_goog = load_catalog("google_search_results.csv", "dl_status", skip_dup=True);  df_goog["catalog"] = "google"
df_pic  = load_catalog("picasso_dora_catalog.csv",  "dl_status");                 df_pic["catalog"]  = "picasso"
df_art  = load_catalog("artnet_catalog.csv",        "status");                    df_art["catalog"]  = "artnet"
df_all  = pd.concat([df_goog, df_pic, df_art], ignore_index=True)

scores = df_all["vision_score_num"]
print(f"Total images:  {len(scores)}")
print(f"Zero scores:   {(scores == 0).sum()}  ({(scores==0).mean()*100:.1f}%)")
print(f"Score > 0:     {(scores > 0).sum()}")
print(f"Max score:     {scores.max():.0f}")

# ── 1. Distribution ──────────────────────────────────────────────────────────
COLORS = {"google": "#4285F4", "picasso": "#EA4335", "artnet": "#FF9800"}

nz = scores[scores > 0]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(nz, bins=80, color="#4285F4", edgecolor="none", log=True)
axes[0].axvline(nz.median(), color="red", linestyle="--", label=f"median={nz.median():.0f}")
axes[0].set_xlabel("Vision score"); axes[0].set_ylabel("Count (log)")
axes[0].set_title(f"Score distribution — non-zero (n={len(nz)})"); axes[0].legend()

sorted_nz = np.sort(nz)
cdf = np.arange(1, len(sorted_nz)+1) / len(scores)
axes[1].plot(sorted_nz, cdf * 100, color="#EA4335")
axes[1].axhline((scores==0).mean()*100, color="gray", linestyle=":",
                label=f"{(scores==0).mean()*100:.0f}% are zero")
axes[1].set_xlabel("Vision score"); axes[1].set_ylabel("% images with score ≤ x")
axes[1].set_title("CDF (all images in denominator)"); axes[1].legend()

plt.tight_layout(); plt.savefig("vision_score_distribution.png", dpi=150); plt.show()

# ── 2. Top-40 bar chart ──────────────────────────────────────────────────────
df_top = (df_all[df_all["vision_score_num"] > 0]
          .sort_values("vision_score_num", ascending=False)
          .head(40).reset_index(drop=True))

def make_label(row):
    t = str(row.get("title", "")).strip()
    y = str(row.get("year",  "")).strip()
    base = (t if t and t != "nan" else Path(str(row.get("local_path",""))).stem)[:52]
    return f"{base} ({y})" if y and y != "nan" else base

df_top["label"] = df_top.apply(make_label, axis=1)

fig, ax = plt.subplots(figsize=(14, 12))
ax.barh(df_top["label"], df_top["vision_score_num"],
        color=df_top["catalog"].map(COLORS))
ax.invert_yaxis()
ax.set_xlabel("Vision score  (full + partial + pages with matching images)")
ax.set_title("Top 40 Dora Maar images — Google Cloud Vision findability score", fontsize=13)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.legend(handles=[mpatches.Patch(color=c, label=k) for k, c in COLORS.items()],
          loc="lower right")

plt.tight_layout(); plt.savefig("vision_histogram.png", dpi=150); plt.show()
print(f"Saved vision_score_distribution.png and vision_histogram.png")


In [ ]:
import squarify
from PIL import Image, ImageOps
import numpy as np

# ── Canvas setup ─────────────────────────────────────────────────────────────
CANVAS_W, CANVAS_H = 5000, 5000
OUT_PATH = "vision_collage.png"

# ── Build rows: sort by area desc (squarify prefers this) ────────────────────
rows = (df_all[["local_path", "vision_score_num", "title"]]
        .assign(area=lambda d: d["vision_score_num"].clip(lower=1))  # zeros → area 1
        .sort_values("area", ascending=False)
        .reset_index(drop=True))

print(f"Total tiles: {len(rows)}  "
      f"(zero-score → white: {(rows['vision_score_num']==0).sum()})")

# ── Squarify layout ───────────────────────────────────────────────────────────
sizes   = rows["area"].tolist()
normed  = squarify.normalize_sizes(sizes, CANVAS_W, CANVAS_H)
rects   = squarify.squarify(normed, 0, 0, CANVAS_W, CANVAS_H)

# ── Render to PIL canvas ─────────────────────────────────────────────────────
canvas = Image.new("RGB", (CANVAS_W, CANVAS_H), "white")

n_rendered = 0
for rect, (_, row) in zip(rects, rows.iterrows()):
    x, y  = int(rect["x"]),  int(rect["y"])
    w, h  = max(int(rect["dx"]), 1), max(int(rect["dy"]), 1)

    if row["vision_score_num"] == 0:
        continue  # leave white

    img_path = str(row["local_path"])
    try:
        img = Image.open(img_path).convert("RGB")
        img = ImageOps.fit(img, (w, h), Image.LANCZOS)  # crop-to-fill
        canvas.paste(img, (x, y))
        n_rendered += 1
    except Exception:
        pass  # leave tile white if image unreadable

    if n_rendered % 100 == 0:
        print(f"  rendered {n_rendered} / {(rows['vision_score_num']>0).sum()} non-zero tiles…")

canvas.save(OUT_PATH)
print(f"\nSaved {OUT_PATH}  ({CANVAS_W}×{CANVAS_H} px, {n_rendered} image tiles)")

# Preview in notebook
display(canvas.resize((800, 800), Image.LANCZOS))
